In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
# --- Our version ---
import os
import sys
from langchain_google_genai import ChatGoogleGenerativeAI
from openai import OpenAI  # Groq and OpenRouter use OpenAI-compatible endpoints

In [3]:
# --- Our version ---
# No single MODEL constant — we support multiple providers, chosen per call.
# No DB_NAME either, since we don't persist to disk (DECISIONS.md D-005).

load_dotenv(override=True)

gemini_api_key = os.getenv('GEMINI_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

In [4]:
import os
os.chdir("..")  # move from notebooks/ up to the pitchwise/ root
print(os.getcwd())  # confirm it's now .../pitchwise

/Users/someshtiwari/Projects/pitchwise


In [5]:
# --- Our version ---
# We don't persist to disk (DECISIONS.md D-005), so instead of reconnecting
# to a saved database, we run our own ingestion pipeline fresh via ingest.py

sys.path.append("..")
from ingest import ingest

vectorstore = ingest()

Loaded 17 documents from knowledge-base
Split into 81 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built: 81 chunks embedded, 384 dimensions


In [6]:
# --- Our version ---
retriever = vectorstore.as_retriever()

# ChatGoogleGenerativeAI is LangChain's wrapper for Gemini — same .invoke()
# interface as ChatOpenAI, so everything downstream stays consistent
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
    google_api_key=gemini_api_key
)

In [7]:
retriever.invoke("Who is Don Bradman?")

[Document(id='17974d89-c365-4dbb-9e76-b16482e13166', metadata={'source': 'knowledge-base/players/don-bradman.md', 'header_1': 'Don Bradman', 'header_2': 'Career Overview', 'doc_type': 'players'}, page_content='Don Bradman is universally regarded as the greatest batter in the history of cricket. He played 52 Test matches for Australia across two decades, spanning the era before and after World War II, and captained Australia for much of his later career.'),
 Document(id='d343b6a7-6b57-41d0-acba-95b8d14ab5ea', metadata={'doc_type': 'players', 'source': 'knowledge-base/players/don-bradman.md', 'header_1': 'Don Bradman'}, page_content='**Full name:** Sir Donald George Bradman\n**Born:** 27 August 1908, Cootamundra, New South Wales, Australia\n**Died:** 25 February 2001, Adelaide, Australia\n**Role:** Batter\n**Batting style:** Right-handed\n**National side:** Australia (1928–1948, retired)'),
 Document(id='20ca08de-5e23-42ef-a5ed-32dddbc15b11', metadata={'source': 'knowledge-base/players/d

In [8]:
llm.invoke("Who is Don Bradman?")

/Users/someshtiwari/Projects/pitchwise/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': '**Sir Donald Bradman** (1908–2001), affectionately known as **"The Don,"** was an Australian cricketer who is universally acknowledged as the greatest batsman in the history of the sport, and arguably the most dominant athlete in the history of any major sport. \n\nHere is what makes him such a legendary figure:\n\n### 1. Statistical Dominance (The 99.94 Average)\nIn Test cricket (the highest international level of the game), a batting average over 50 is considered the mark of an all-time great. Bradman retired with a career Test batting average of **99.94**. \n* The second-highest average in history (for players with a reasonable number of innings) is around 61.8. \n* Bradman’s statistical superiority over his peers is greater than that of Michael Jordan in basketball, Wayne Gretzky in hockey, or Pelé in football.\n\n### 2. The Famous Final Innings\nIn his very last Test match in 1948, Bradman needed just **4 runs** to finish his career wit

In [9]:
# --- Our version ---
SYSTEM_PROMPT_TEMPLATE = """
You are Pitchwise, a knowledgeable, friendly cricket assistant.
You are chatting with a user about cricket — players, formats, tournaments, and the laws of the game.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [10]:
# --- Our version ---
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [11]:
answer_question("What is Don Bradman's exact career Test batting average?", [])

/Users/someshtiwari/Projects/pitchwise/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text',
  'text': "Sir Don Bradman's exact career Test batting average is **99.94**. \n\nIt is widely considered the most famous statistic in cricket—and perhaps all of sports! Had he scored just 4 runs in his final Test innings instead of getting out for a duck, he would have finished with an average of exactly 100.",
  'extras': {'signature': 'EowGCokGARFNMg805KPEtqBY5nS3rPbWZByqxhtFOfvXTkMKcpOgmlVM0oLwM+ZEE4ImFnOJXFh5B9pvhDc1WPLQb9LMDSgsdsxesCqgZU9AeDewaUDcbB9sO+ENo4GYLQ8o+Lia3v3Zu0h/HpSoKX5quLVP1q6D+wtWTNiCu1+jVk2WBtBc9lFfSxglYfv7WxVHLpLkVP8YJxK7VdlrRg24iORl0bsW3naoh7tj0tA/OV9pMaOO7p7GQedlSBLYT3L1dVmwUA3lF46FAXhL5ho23imyb5THyvFRND3nmG5j95evAMb+nXrlz7lNO7Wg+X0ElDDNsWbI2YcHruTUDijZCH2kKOkbuo0/Dd2dFV5gsbqvYgm0GVQBScA02w2qSaVi60xva3KrITZVl0btMV1yI9FUi7mEa+Mhkm4OfQZxoOsZ5uDk5hlDbfcVhSnhIxv6SNPplQn5yK3UxobzUq766s2KuNs/UT9sUOAHNKyCctDVSMCkq8onitqOrPOxiWaLfy7I4CB9YAHLhYupJ9Ka9z15vjfv+G4NL/JesIKwK19WU1iCJQ6MALMF14SV86DcXS8c/y+7ZltknjOYWKa91gW4GSb0LlYUhPmBS4XpexYvzip5KUZ1Lis8RAMvW0

In [12]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
import time

start = time.time()
response = answer_question("What is Don Bradman's exact career Test batting average?", [])
elapsed = time.time() - start
print(f"Took {elapsed:.1f} seconds")

/Users/someshtiwari/Projects/pitchwise/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Took 4.3 seconds


In [14]:
import time

# Time retrieval alone
start = time.time()
docs = retriever.invoke("What is Don Bradman's exact career Test batting average?")
print(f"Retrieval: {time.time() - start:.1f}s")

# Time the LLM call alone (using context we just retrieved)
context = "\n\n".join(doc.page_content for doc in docs)
system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

start = time.time()
response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content="What is Don Bradman's exact career Test batting average?")])
print(f"LLM call: {time.time() - start:.1f}s")

Retrieval: 0.0s


/Users/someshtiwari/Projects/pitchwise/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


LLM call: 3.7s


In [15]:
# --- Testing Groq for the same query ---
from openai import OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

start = time.time()
groq_response = groq.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "What is Don Bradman's exact career Test batting average?"}
    ]
)
print(f"Groq call: {time.time() - start:.1f}s")
print(groq_response.choices[0].message.content)

Groq call: 1.0s
Don Bradman’s career Test batting average is **99.94**. This figure comes from his 6,996 runs in 52 Tests and remains the highest ever recorded for a player with a substantial number of innings.


In [16]:
# --- Full provider/model speed comparison ---
import time

question = "What is Don Bradman's exact career Test batting average?"
# system_prompt already built from earlier retrieval — reusing it here

candidates = {
    "gemini-3.6-flash": ("gemini", "gemini-3.6-flash"),
    "gemini-3.5-flash-lite": ("gemini", "gemini-3.5-flash-lite"),
    "groq: gpt-oss-120b": ("groq", "openai/gpt-oss-120b"),
    "groq: gpt-oss-20b": ("groq", "openai/gpt-oss-20b"),
    "groq: qwen3.8-27b": ("groq", "qwen/qwen3.8-27b"),
    "openrouter: minimax-m3": ("openrouter", "minimax/minimax-m3:free"),
    "openrouter: glm-5.2": ("openrouter", "z-ai/glm-5.2:free"),
}

groq_client = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

speed_results = {}

for label, (provider, model) in candidates.items():
    try:
        start = time.time()

        if provider == "gemini":
            temp_llm = ChatGoogleGenerativeAI(model=model, temperature=0, google_api_key=gemini_api_key)
            response = temp_llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
            answer_text = response.content

        elif provider == "groq":
            response = groq_client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
            )
            answer_text = response.choices[0].message.content

        elif provider == "openrouter":
            response = openrouter_client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]
            )
            answer_text = response.choices[0].message.content

        elapsed = time.time() - start
        speed_results[label] = {"time": elapsed, "answer": answer_text}
        print(f"{label}: {elapsed:.1f}s")

    except Exception as e:
        print(f"{label}: FAILED — {e}")
        speed_results[label] = {"time": None, "answer": None}

print("\n--- Ranked by speed ---")
ranked = sorted(
    [(k, v["time"]) for k, v in speed_results.items() if v["time"] is not None],
    key=lambda x: x[1]
)
for label, t in ranked:
    print(f"{t:.1f}s — {label}")

/Users/someshtiwari/Projects/pitchwise/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


gemini-3.6-flash: 5.5s


/Users/someshtiwari/Projects/pitchwise/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


gemini-3.5-flash-lite: 1.2s
groq: gpt-oss-120b: 0.7s
groq: gpt-oss-20b: 0.7s
groq: qwen3.8-27b: 0.4s
openrouter: minimax-m3: 2.4s
openrouter: glm-5.2: FAILED — Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'z-ai/glm-5.2:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Decart', 'is_byok': False, 'provider_error_code': 'upstream_429', 'limit_source': 'upstream_provider_shared_pool', 'remedy_hint': 'Retry shortly, add your own provider key (https://openrouter.ai/settings/integrations), or route to another provider with provider routing: https://openrouter.ai/docs/features/provider-routing', 'retry_after_seconds': 5, 'retry_after_seconds_raw': 5, 'headers': {'Retry-After': '5'}}}, 'user_id': 'user_3IivxsJc3BhajoYB0c7NwBwlwe1'}

--- Ranked by speed ---
0.4s — groq: qwen3.8-27b
0.7s — groq: gpt-oss-20b
0.7s —

In [17]:
# --- Testing the fallback mechanism (DECISIONS.md D-008) ---
import answer
from ingest import ingest

# Build a vector store to test against (if you don't already have one
# named `store` in this notebook session)
store = ingest()

real_groq_key = answer.GROQ_API_KEY
answer.GROQ_API_KEY = "intentionally-invalid-key-for-testing"

test_question = "What is Don Bradman's exact career Test batting average?"

result = answer.answer_question(
    store,
    test_question,
    model_label="Groq: qwen3.8-27b (fastest, default)"
)

print(f"Q: {test_question}")
print(f"A: {result}")

answer.GROQ_API_KEY = real_groq_key
print("\nGroq key restored.")

Loaded 17 documents from knowledge-base
Split into 81 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built: 81 chunks embedded, 384 dimensions


/Users/someshtiwari/Projects/pitchwise/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q: What is Don Bradman's exact career Test batting average?
A: Don Bradman's exact career Test batting average is **99.94**! 

It's easily the most famous statistic in all of cricket—and by a very wide margin, the highest career Test average of any player with a significant number of innings. In fact, statisticians often cite it as the single greatest statistical outlier in any major sport!

Groq key restored.
